## postcode price predictor

interactive front end for `src/predict.py` -- same model and features as `02_modeling.ipynb`, just packaged for a single "what would this property sell for" query instead of evaluating the model as a whole.

requires `python -m src.predict --train` to have been run at least once (already done -- it saved a trained model + a snapshot of each area's recent price data to `data/processed/`). re-run `--train` only if the underlying sales data changes.

inputs:
- **postcode** -- any UK postcode, quoted
- **property_type** -- `F` flat, `S` semi-detached, `T` terraced, `D` detached
- **duration** -- `F` freehold, `L` leasehold
- **old_new** -- optional, defaults to `N` (existing property), or `Y` for new build

## what this prediction is (and isn't) based on

this model only knows what's in land registry sale records + postcode coordinates:

- property type, tenure (freehold/leasehold), new-build flag
- distance from central london
- the area's own recent sale prices/volume, and its immediate neighbours' (the hex features -- see `01_eda.ipynb`/`02_modeling.ipynb`)

it does **not** observe, and cannot separately account for:

- anything about the *specific* property -- condition, size, bedroom count, garden, parking, renovation state, exact street/address
- crime rate, school catchment, transit links, green space, or any other named local amenity -- none of these are measured directly. they only show up *indirectly*, to the extent they are already baked into an area's recent sale prices (the hex trailing price is a proxy for "whatever locally drives value here," not a list of named factors -- see PLAN.md's hex-price assumption)

scope limits:

- trained only on greater london sales, 2023-01 to 2025-12 -- a postcode outside that radius raises an error rather than extrapolating (demonstrated below)
- "now" for this model means the end of the training window (2025-12-31), not today's actual market -- prices are not refreshed in real time
- this is a point estimate from a model that explains roughly 57-61% of price variation (R² in `02_modeling.ipynb`), with a typical error in the region of 30% of price -- read it as a ballpark for a *type* of property in an *area*, not a valuation of one specific house
- `old_new` silently defaults to `N` (existing property) if you don't pass it

In [1]:
import sys
sys.path.append('..')
from src.predict import predict_price

/Users/cam/Documents/datascience/cv_projects/house_price_prediction/house_price_prediction_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
f"£{predict_price('SW4 6BZ', 'F', 'L'):,.0f}"

'£496,903'

comparing locations only makes sense holding property type and tenure constant -- a flat in one postcode against a house in another mixes the location effect with the type effect and the comparison stops meaning anything.

In [3]:
for postcode in ['SW4 6BZ', 'CR5 2JD']:
    print(postcode, '->', f"£{predict_price(postcode, 'T', 'F'):,.0f}")

SW4 6BZ -> £994,198


CR5 2JD -> £499,995


postcodes too far outside greater london raise an error rather than silently returning a number -- the model was never trained on anything past ~32km from central london.

In [4]:
try:
    predict_price('M1 1AE', 'F', 'L')
except ValueError as e:
    print('error:', e)

error: postcode 'M1 1AE' is outside the trained region (too far from central london)
